This Jupyter Notebook demonstrates a comprehensive pipeline for evaluating various **Information Retrieval (IR)** architectures using the classic **Cranfield dataset**. It progresses from traditional keyword-based methods to modern neural "Cross-Encoder" re-ranking strategies.



## Cell 1: Environment Setup
This cell installs the necessary Python libraries to build and evaluate the IR models:
* `scikit-learn`: Used for traditional Vector Space Models (TF-IDF).
* `rank_bm25`: A specialized library for the BM25 probabilistic ranking algorithm.
* `sentence-transformers`: Provides the framework for Bi-Encoders and Cross-Encoders (state-of-the-art neural IR).
* `numpy`: Handles numerical operations and array manipulations.

In [1]:
!pip install scikit-learn rank_bm25 sentence-transformers numpy


## Cell 2: Library Imports
This section imports the core modules required for the script:
* `ir_datasets`: A gateway to standard IR benchmarks (like Cranfield).
* `pytrec_eval`: The industry-standard tool for computing IR metrics (Precision, Recall, etc.).
* `torch`: The underlying engine for the neural network models.
* `warnings`: Used here to suppress non-critical logs to keep the output clean.




In [37]:
import numpy as np
import ir_datasets
import pytrec_eval
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
import warnings
import torch

warnings.filterwarnings('ignore')

## Cell 3: Data Loading & Preprocessing
This cell prepares the **Cranfield dataset**, which contains 1,400 documents about aerodynamics.
* **Corpus Creation**: It combines document titles and text into a single searchable string for each entry.
* **Query & Qrels**: It loads the search queries and the "Ground Truth" (Qrels), which tells us which documents are actually relevant to which queries.
* **Test Constraints**: To ensure the code runs quickly, it limits the initial evaluation to the first 5 queries.



In [38]:
print("Loading Cranfield Dataset...")
dataset = ir_datasets.load("cranfield")

doc_ids = []
corpus = []
for doc in dataset.docs_iter():
    doc_ids.append(doc.doc_id)
    corpus.append(f"{doc.title} {doc.text}")

TOTAL_DOCS = len(corpus)

queries = {query.query_id: query.text for query in dataset.queries_iter()}

qrels_trec = {}
for qrel in dataset.qrels_iter():
    if qrel.relevance > 0: # Only store actually relevant docs
        if qrel.query_id not in qrels_trec:
            qrels_trec[qrel.query_id] = {}
        qrels_trec[qrel.query_id][qrel.doc_id] = int(qrel.relevance)

# Use just 5 queries for quick testing. Change to list(qrels_trec.keys()) for all.
eval_query_ids = list(qrels_trec.keys())[:5]
K = 10

Loading Cranfield Dataset...


## Cell 4: Model Indexing & Initialization
Here, the notebook prepares the "engines" for five distinct IR strategies:
1.  **Boolean**: Simplistic term-matching (tokenization).
2.  **TF-IDF**: Builds a Vector Space Model based on term frequency and inverse document frequency.
3.  **BM25**: Initializes the probabilistic model used by modern search engines like Elasticsearch.
4.  **Bi-Encoder**: Loads a pre-trained Transformer (`all-MiniLM-L6-v2`) to convert all documents into "dense vectors" (embeddings) that capture semantic meaning.
5.  **Cross-Encoder**: Loads a highly accurate re-ranking model (`ms-marco-MiniLM-L-6-v2`) that compares query-document pairs directly.



In [39]:
print("Indexing Corpus...")

# 1. Boolean
def tokenize(text):
    return set(text.lower().split())
boolean_corpus = [tokenize(doc) for doc in corpus]

# 2. TF-IDF
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(corpus)

# 3. BM25
bm25_corpus = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(bm25_corpus)

# 4. Bi-Encoder
print("Loading Bi-Encoder...")
bi_encoder = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = bi_encoder.encode(corpus, convert_to_tensor=True)

# 5. Cross-Encoder
print("Loading Cross-Encoder...")
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

Indexing Corpus...
Loading Bi-Encoder...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading Cross-Encoder...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Cell 5: Retrieval Functions
This cell defines the logic for how each model searches the corpus:
* `retrieve_boolean`: Counts how many query words appear in a document.
* `retrieve_tfidf`: Uses **Cosine Similarity** to find documents with similar word distributions to the query.
* `retrieve_bi_encoder`: Compares the query embedding to document embeddings using vector similarity.
* **The Pipeline**: `retrieve_cross_encoder_pipeline` uses a two-stage approach. It first retrieves 100 candidates using BM25 (cheap) and then re-ranks them using the powerful Cross-Encoder (expensive but accurate).



In [40]:
def retrieve_boolean(query):
    query_tokens = tokenize(query)
    scores = [len(query_tokens.intersection(doc)) for doc in boolean_corpus]
    top_indices = np.argsort(scores)[::-1]
    return [doc_ids[i] for i in top_indices]

def retrieve_tfidf(query):
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, tfidf_matrix)[0]
    top_indices = np.argsort(scores)[::-1]
    return [doc_ids[i] for i in top_indices]

def retrieve_bm25(query):
    q_tokens = query.lower().split()
    scores = bm25.get_scores(q_tokens)
    top_indices = np.argsort(scores)[::-1]
    return [doc_ids[i] for i in top_indices]

def retrieve_bi_encoder(query):
    q_emb = bi_encoder.encode(query, convert_to_tensor=True)
    scores = torch.nn.functional.cosine_similarity(q_emb, doc_embeddings)
    top_indices = torch.argsort(scores, descending=True).cpu().numpy()
    return [doc_ids[i] for i in top_indices]

def retrieve_cross_encoder_pipeline(query):
    q_tokens = query.lower().split()
    bm25_scores = bm25.get_scores(q_tokens)
    top_100_idx = np.argsort(bm25_scores)[::-1][:100]

    pairs = [[query, corpus[i]] for i in top_100_idx]
    ce_scores = cross_encoder.predict(pairs)

    reranked_local_idx = np.argsort(ce_scores)[::-1]
    final_top_indices = [top_100_idx[i] for i in reranked_local_idx]
    return [doc_ids[i] for i in final_top_indices]

## Cell 6: Evaluation & Metrics
The final cell executes the experiment and produces the comparison table.
* **Metrics Calculated**:
    * **Precision**: Percentage of the top 10 results that are relevant.
    * **Recall**: Percentage of all relevant documents that were found in the top 10.
    * **F1-Score**: The harmonic mean of Precision and Recall.
    * **Fallout**: The rate at which irrelevant documents are retrieved (lower is better).
* **Process**: It iterates through every model, processes the sample queries, and uses `pytrec_eval` to score the results against the ground truth.

In [42]:
models = {
    "1. Boolean (Term Count)   ": retrieve_boolean,
    "2. TF-IDF (Vector Space)  ": retrieve_tfidf,
    "3. BM25 (Probabilistic)   ": retrieve_bm25,
    "4. Bi-Encoder (Dense)     ": retrieve_bi_encoder,
    "5. Cross-Encoder (Re-rank)": retrieve_cross_encoder_pipeline
}

# Initialize pytrec_eval with desired standard metrics
evaluator = pytrec_eval.RelevanceEvaluator(qrels_trec, {'P_10', 'recall_10'})

print(f"\nEvaluating models on {len(eval_query_ids)} queries @ Top-{K}...\n")
print(f"{'Model':<28} | {'Precision':<8} | {'Recall':<9} | {'F1':<8} | {'Fallout':<10}")
print("-" * 75)

for model_name, retrieve_func in models.items():
    trec_run = {}

    # Generate the search results format required by pytrec_eval
    for q_id in eval_query_ids:
        query_text = queries[q_id]
        ranked_docs = retrieve_func(query_text)[:K]

        # pytrec_eval requires {doc_id: score}. We use reverse rank (10, 9, 8...) as the score.
        trec_run[q_id] = {doc_id: float(K - i) for i, doc_id in enumerate(ranked_docs)}

    # Run pytrec_eval
    trec_results = evaluator.evaluate(trec_run)

    avg_p = 0.0
    avg_r = 0.0
    avg_f1 = 0.0
    avg_fallout = 0.0

    for q_id, metrics in trec_results.items():
        p = metrics['P_10']
        r = metrics['recall_10']

        # 1. Calculate F1
        f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0.0

        # 2. Calculate Fallout
        true_positives = round(p * K) # E.g., 0.4 * 10 = 4 actual relevant docs retrieved
        false_positives = K - true_positives # 10 - 4 = 6 garbage docs retrieved

        # Safeguard: if the query isn't in ground truth (rare, but good practice)
        total_relevant = len(qrels_trec.get(q_id, {}))
        total_irrelevant = TOTAL_DOCS - total_relevant

        fallout = false_positives / total_irrelevant if total_irrelevant > 0 else 0.0

        # Add to running totals
        avg_p += p
        avg_r += r
        avg_f1 += f1
        avg_fallout += fallout

    N = len(trec_results)
    if N > 0:
        print(f"{model_name:<28} | {avg_p/N:.4f}   | {avg_r/N:.4f}      | {avg_f1/N:.4f}   | {avg_fallout/N:.5f}")


Evaluating models on 5 queries @ Top-10...

Model                        | Precision | Recall    | F1       | Fallout   
---------------------------------------------------------------------------
1. Boolean (Term Count)      | 0.2000   | 0.2714      | 0.1796   | 0.00576
2. TF-IDF (Vector Space)     | 0.3800   | 0.5190      | 0.3568   | 0.00447
3. BM25 (Probabilistic)      | 0.3400   | 0.3940      | 0.3013   | 0.00475
4. Bi-Encoder (Dense)        | 0.4000   | 0.5595      | 0.3883   | 0.00432
5. Cross-Encoder (Re-rank)   | 0.3600   | 0.5190      | 0.3410   | 0.00461
